In [2]:
from pyspark.sql import SparkSession
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("Ecom_Hive_Project") \
    .config("spark.driver.memory", "6g") \
    .enableHiveSupport() \
    .getOrCreate()

print("新 SparkSession 创建成功")

26/04/17 22:54:52 WARN Utils: Your hostname, LAPTOP-KVJ76CML resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/17 22:54:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 22:54:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


新 SparkSession 创建成功


In [3]:
spark.sql("SHOW DATABASES").show()

26/04/17 22:54:55 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/04/17 22:54:55 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


+--------------+
|     namespace|
+--------------+
|       default|
|  ecommerce_db|
|       test_db|
|wsl_local_test|
+--------------+



In [4]:
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_db")
spark.sql("USE ecommerce_db")
print("已切换到数据库: ecommerce_db")

已切换到数据库: ecommerce_db


26/04/17 22:54:57 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


In [5]:
base = r"/home/lst/my-spark/star_schema"

In [6]:
spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS fact_order_items (
    item_id             STRING COMMENT '订单明细唯一ID',
    increment_id        STRING COMMENT '订单号',
    created_at          TIMESTAMP COMMENT '下单时间',
    price               DECIMAL(10,2) COMMENT '商品单价',
    qty_ordered         INT COMMENT '购买数量',
    total_value         DECIMAL(12,2) COMMENT '行总金额 = price * qty',
    grand_total         DECIMAL(12,2) COMMENT '订单总金额',
    discount_amount     DECIMAL(10,2) COMMENT '折扣金额',
    status              STRING COMMENT '订单状态',
    payment_method      STRING COMMENT '支付方式',
    sku                 STRING COMMENT '商品SKU',
    customer_id         STRING COMMENT '客户ID',
    category_name_1     STRING COMMENT '一级品类'
)
COMMENT '电商订单事实表 - 订单明细粒度（星型模型核心事实表）'
PARTITIONED BY (year INT COMMENT '年份', month INT COMMENT '月份')
STORED AS PARQUET
LOCATION '{base}fact_order_items'
TBLPROPERTIES (
    'parquet.compression' = 'SNAPPY'
);
""")

print("fact_order_items（事实表）已创建")

26/04/17 22:55:30 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/04/17 22:55:30 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/04/17 22:55:30 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/04/17 22:55:30 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


✅ fact_order_items（事实表）已创建


In [7]:
spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS dim_products (
    sku                 STRING COMMENT '商品SKU',
    category_name_1     STRING COMMENT '一级品类'
)
COMMENT '商品维度表'
STORED AS PARQUET
LOCATION '{base}dim_products'
""")

spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS dim_customers (
    customer_id         STRING COMMENT '客户唯一ID',
    customer_since      STRING COMMENT '客户注册时间'
)
COMMENT '客户维度表'
STORED AS PARQUET
LOCATION '{base}dim_customers'
""")

spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS dim_time (
    created_at          TIMESTAMP COMMENT '时间戳',
    year                INT COMMENT '年份',
    month               INT COMMENT '月份'
)
COMMENT '时间维度表'
STORED AS PARQUET
LOCATION '{base}dim_time'
""")

spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS dim_status (
    status              STRING COMMENT '订单状态'
)
COMMENT '订单状态维度表'
STORED AS PARQUET
LOCATION '{base}dim_status'
""")

print("所有维度表已创建")

# 刷新事实表分区
spark.sql("MSCK REPAIR TABLE fact_order_items")

print("\n 所有 Hive 表创建完成！")

# 验证
spark.sql("SHOW TABLES").show()

所有维度表已创建

 所有 Hive 表创建完成！
+------------+----------------+-----------+
|   namespace|       tableName|isTemporary|
+------------+----------------+-----------+
|ecommerce_db|   dim_customers|      false|
|ecommerce_db|    dim_products|      false|
|ecommerce_db|      dim_status|      false|
|ecommerce_db|        dim_time|      false|
|ecommerce_db|fact_order_items|      false|
+------------+----------------+-----------+



In [8]:
# 验证表是否可用
print("=== Hive 表行数统计 ===")
spark.sql("""
SELECT 'fact_order_items' as table_name, count(*) as row_count FROM fact_order_items
UNION ALL
SELECT 'dim_products', count(*) FROM dim_products
UNION ALL
SELECT 'dim_customers', count(*) FROM dim_customers
UNION ALL
SELECT 'dim_time', count(*) FROM dim_time
UNION ALL
SELECT 'dim_status', count(*) FROM dim_status
""").show()

=== Hive 表行数统计 ===
+----------------+---------+
|      table_name|row_count|
+----------------+---------+
|fact_order_items|        0|
|    dim_products|        0|
|   dim_customers|        0|
|        dim_time|        0|
|      dim_status|        0|
+----------------+---------+



In [25]:
spark.sql("USE ecommerce_db")

print("=== 当前所有表 ===")
spark.sql("SHOW TABLES").show()

print("\n=== fact_order_items 表的结构（看字段名） ===")
spark.sql("DESCRIBE fact_order_items").show(truncate=False)

print("\n=== dim_customers 表的结构 ===")
spark.sql("DESCRIBE dim_customers").show(truncate=False)

=== 当前所有表 ===
+------------+-------------+-----------+
|   namespace|    tableName|isTemporary|
+------------+-------------+-----------+
|ecommerce_db|dim_customers|      false|
|ecommerce_db| dim_products|      false|
|ecommerce_db|   dim_status|      false|
|ecommerce_db|     dim_time|      false|
+------------+-------------+-----------+


=== fact_order_items 表的结构（看字段名） ===


AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `fact_order_items` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 9;
'DescribeRelation false, [col_name#442, data_type#443, comment#444]
+- 'UnresolvedTableOrView [fact_order_items], DESCRIBE TABLE, true


In [26]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("Rebuild_Hive_Tables_Clean") \
    .config("spark.driver.memory", "6g") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

spark.sql("USE ecommerce_db")

base = "/home/lst/my-spark/star_schema/"   # ←←← 请确认并修改成你的实际路径

print("开始从零重建所有 Hive 表...")

# 删除所有旧表，避免残留问题
tables = ["fact_order_items", "dim_products", "dim_customers", "dim_time", "dim_status"]
for t in tables:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

print("旧表已全部删除")

开始从零重建所有 Hive 表...
旧表已全部删除


In [27]:
spark.sql("SHOW TABLES").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [28]:
spark.sql(f"""
CREATE EXTERNAL TABLE fact_order_items (
    item_id           STRING,
    increment_id      STRING,
    created_at        TIMESTAMP,
    price             DOUBLE,
    qty_ordered       DOUBLE,
    total_value       DOUBLE,
    grand_total       DOUBLE,
    discount_amount   DOUBLE,
    status            STRING,
    payment_method    STRING,
    sku               STRING,
    `Customer ID`     STRING,
    category_name_1   STRING
)
PARTITIONED BY (year INT, month INT)
STORED AS PARQUET
LOCATION '{base}fact_order_items'
""")

# 2. 维度表
spark.sql(f"CREATE EXTERNAL TABLE dim_products STORED AS PARQUET LOCATION '{base}dim_products' AS SELECT * FROM parquet.`{base}dim_products`")
spark.sql(f"CREATE EXTERNAL TABLE dim_customers STORED AS PARQUET LOCATION '{base}dim_customers' AS SELECT * FROM parquet.`{base}dim_customers`")
spark.sql(f"CREATE EXTERNAL TABLE dim_time STORED AS PARQUET LOCATION '{base}dim_time' AS SELECT * FROM parquet.`{base}dim_time`")
spark.sql(f"CREATE EXTERNAL TABLE dim_status STORED AS PARQUET LOCATION '{base}dim_status' AS SELECT * FROM parquet.`{base}dim_status`")

print("✅ 所有 5 张表已重新创建")

# 刷新分区
spark.sql("MSCK REPAIR TABLE fact_order_items")

print("\n=== 当前所有表 ===")
spark.sql("SHOW TABLES").show()

print("\n=== fact_order_items 表结构 ===")
spark.sql("DESCRIBE fact_order_items").show(truncate=False)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/1st/my-spark/star_schema/dim_products.